In [ ]:
!pip install librosa transformers datasets torchaudio openai-whisper

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd

import librosa
import librosa.display

import matplotlib.pyplot as plt
import seaborn as sns

import whisper

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.utils import class_weight

from tensorflow.keras.utils import to_categorical

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras import layers, models

In [ ]:
zip_path = "/content/drive/MyDrive/Multimodal_Emotion_Recognition/archive (1).zip"

extract_path = "/content/ravdess"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset Extracted Successfully")

In [ ]:
emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

In [ ]:
audio_files = []
labels = []

dataset_root = "/content/ravdess/audio_speech_actors_01-24"

for actor_folder in os.listdir(dataset_root):

    actor_path = os.path.join(dataset_root, actor_folder)

    for file in os.listdir(actor_path):

        if file.endswith(".wav"):

            full_path = os.path.join(actor_path, file)

            emotion_code = file.split("-")[2]

            emotion_label = emotion_map[emotion_code]

            audio_files.append(full_path)

            labels.append(emotion_label)

print("Total Audio Files:", len(audio_files))

In [ ]:
def extract_mel_spectrogram(file_path, n_mels=128):
    #loading audio
    # y is audio signal and sr is the sample rate(no of data points that are collected per second.)

    y, sr = librosa.load(file_path,sr=22050)

    # target_length is no of data points per 3 seconds

    target_length = sr * 3

    #padding so that length of y becomes target_length

    if len(y) < target_length:
        y = np.pad(y, (0, target_length - len(y)))
    else:
        y = y[:target_length]

     #converting  waveforms into mel-spectrograms

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)

    return librosa.power_to_db(mel, ref=np.max)

In [ ]:
X_audio = []
y_audio = []

for file_path, label in zip(audio_files, labels):

    mel_spec = extract_mel_spectrogram(file_path)

    X_audio.append(mel_spec)

    y_audio.append(label)

print("Feature Extraction Completed")


In [ ]:
X_audio = np.array(X_audio)

y_audio = np.array(y_audio)

print(X_audio.shape)
print(y_audio.shape)

In [ ]:
X_audio = X_audio[..., np.newaxis]

print(X_audio.shape)
encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y_audio)

print(encoder.classes_)

y_onehot = to_categorical(y_encoded)

print(y_onehot.shape)
X_audio_train, X_audio_test, y_train, y_test = train_test_split(
    X_audio,
    y_onehot,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(X_audio_train.shape)
print(X_audio_test.shape)

In [ ]:
# 1. Load Whisper Model (using 'tiny' for speed on a deadline)
# This is to transcribe clips to text

model_whisper = whisper.load_model("tiny")

def get_transcripts(file_paths):

    transcripts = []

    for path in file_paths:

        result = model_whisper.transcribe(path, fp16=False)

        transcripts.append(result["text"])

    return transcripts
all_texts = get_transcripts(audio_files)

print(all_texts[:5])
# 2. Tokenization and Padding
# Assuming 'all_texts' is your list of transcribed strings
tokenizer = Tokenizer(num_words=1000, lower=True)

tokenizer.fit_on_texts(all_texts)

sequences = tokenizer.texts_to_sequences(all_texts)

# Pad to ensure all text inputs are the same length (e.g., 20 words)

max_text_length = 20

X_text = pad_sequences(
    sequences,
    maxlen=max_text_length,
    padding='post'
)

print(X_text.shape)

In [ ]:
X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    X_text,
    y_onehot,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [ ]:
def build_audio_cnn(input_shape):

    model = models.Sequential([

        layers.Input(shape=input_shape), # e.g., (128, 130, 1)

        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),

        layers.Dense(128, activation='relu'), # "Bottleneck" layer for later fusion

        layers.Dense(8, activation='softmax') # 8 Emotions
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


In [ ]:

audio_model = build_audio_cnn(X_audio_train.shape[1:])

audio_model.summary()

In [ ]:

history_audio = audio_model.fit(
    X_audio_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)

In [ ]:
#Text RNN
def build_text_rnn(vocab_size, max_len):

    model = models.Sequential([

        layers.Input(shape=(max_len,)),

        layers.Embedding(input_dim=vocab_size, output_dim=64),

        layers.LSTM(64, return_sequences=False),

        layers.Dense(64, activation='relu'), # "Bottleneck" layer for later fusion

        layers.Dense(8, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


In [ ]:
text_model = build_text_rnn(
    vocab_size=1000,
    max_len=max_text_length
)

text_model.summary()

In [ ]:
history_text = text_model.fit(
    X_text_train,
    y_text_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)


In [ ]:
def late_fusion_predict(audio_model, text_model, X_audio_test, X_text_test, weight_audio=0.7):

    # 1. Get probability scores (0.0 to 1.0) from both models

    audio_probs = audio_model.predict(X_audio_test)

    text_probs = text_model.predict(X_text_test)

    # 2. Weighted Averaging
    # (0.7 * Audio confidence) + (0.3 * Text confidence)

    final_probs = (weight_audio * audio_probs) + ((1 - weight_audio) * text_probs)

    # 3. Take the index of the highest probability

    final_predictions = np.argmax(final_probs, axis=1)

    return final_predictions

In [ ]:
y_pred = audio_model.predict(X_audio_test)

y_pred_classes = np.argmax(y_pred, axis=1)

y_true = np.argmax(y_test, axis=1)

In [ ]:

emotion_labels = [
    'Neutral',
    'Calm',
    'Happy',
    'Sad',
    'Angry',
    'Fearful',
    'Disgust',
    'Surprised'
]

print(
    classification_report(
        y_true,
        y_pred_classes,
        target_names=emotion_labels
    )
)

In [ ]:
plt.figure(figsize=(10,8))

cm = confusion_matrix(y_true, y_pred_classes)

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Reds',
    xticklabels=emotion_labels,
    yticklabels=emotion_labels
)

plt.title('Emotion Confusion Matrix')

plt.ylabel('Actual Emotion')

plt.xlabel('Predicted Emotion')

plt.show()

In [ ]:
fusion_predictions = late_fusion_predict(
    audio_model,
    text_model,
    X_audio_test,
    X_text_test,
    weight_audio=0.7
)

fusion_true = np.argmax(y_test, axis=1)

In [ ]:
print(
    classification_report(
        fusion_true,
        fusion_predictions,
        target_names=emotion_labels
    )
)

In [ ]:
fusion_cm = confusion_matrix(
    fusion_true,
    fusion_predictions
)

plt.figure(figsize=(10,8))

sns.heatmap(
    fusion_cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=emotion_labels,
    yticklabels=emotion_labels
)

plt.title('Late Fusion Confusion Matrix')

plt.ylabel('Actual Emotion')

plt.xlabel('Predicted Emotion')

plt.show()